# Would you trust this model?
**Imaging & CNNs workshop · Laptop part 3 · teams of 2 to 4 · about 22 minutes**

## The situation

A hospital wants a model that looks at a chest X-ray and reports which of 14 findings it shows. Someone asked an AI assistant to write the training code, and the code further down is what came back.

The code runs without any errors. At the end it reports that the model is correct about **95%** of the time on the test set, and it says that the model is ready for the hospital. The hospital would like to start using it next week, but first they have asked your team to review the code.

## What you are looking for

The code contains **four problems**. None of them is a typo, and none of them makes the code crash. Each one is a *decision* that was made badly. Every problem either makes the 95% look better than the model really is, or makes the model itself worse. You have seen all four ideas earlier today.

## How this part works

| Phase | What you do | Time |
|---|---|---|
| 1. Review without AI | Read the code as a team and fill in the worksheet. Please don't use AI tools yet. We want to find out how much you can spot with what you learned today. | 12 min |
| 2. Review with AI | Ask Claude or ChatGPT to review the same code, and compare its answers with yours. | 8 min |
| 3. Verdict | Decide as a team whether the hospital can use this model. | 2 min |

You don't have to run the code to find the problems, because reading it is the exercise. If you want to test a suspicion, you can run it: training takes about two minutes on a GPU.

💬 **A good way to review as a team:** one person reads the code out loud, step by step, and explains in their own words what each line does. The others interrupt as soon as something sounds odd.

In [ ]:
#@title ▶️ Run this cell first. It loads the X-rays and two helper functions (about 1 minute).
!pip -q install medmnist
import copy, numpy as np, torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from medmnist import ChestMNIST

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def load(split):
    ds = ChestMNIST(split=split, download=True, size=28, root='.')
    x = torch.tensor(ds.imgs, dtype=torch.float32).unsqueeze(1) / 255
    return x, torch.tensor(ds.labels, dtype=torch.float32)

# The dataset comes with its own split into training, validation and test X-rays.
train_x, train_y = load('train')
val_x, val_y = load('val')
test_x, test_y = load('test')
mean, std = train_x.mean(), train_x.std()
train_x, val_x, test_x = [(x - mean) / std for x in (train_x, val_x, test_x)]

def train_one_epoch(model, X, Y, loss_fn, optimizer, batch_size=128):
    # Show the model every X-ray in X once, in random batches of 128, and update the parameters after each batch.
    model.train()
    order = torch.randperm(len(X))
    for i in range(0, len(X), batch_size):
        batch = order[i:i + batch_size]
        loss = loss_fn(model(X[batch].to(device)), Y[batch].to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

@torch.no_grad()
def predict(model, X):
    # Return the model's 14 outputs for every X-ray in X, as one table with a row per X-ray.
    model.eval()
    return torch.cat([model(X[i:i + 2048].to(device)).cpu() for i in range(0, len(X), 2048)])

print(f'✅ Ready. The dataset has {len(train_x):,} training, {len(val_x):,} validation and {len(test_x):,} test X-rays.')

---
## The code the AI wrote

The code has four steps. It uses the two helper functions from the setup cell:

- `train_one_epoch(model, X, Y, ...)` shows the model every X-ray in `X` once and adjusts the model's parameters as it goes.
- `predict(model, X)` returns the model's 14 outputs for every X-ray in `X`.

`X` always holds X-ray images, and `Y` holds the matching labels. Each label is a row of 14 zeros and ones, with a 1 for every finding a radiologist saw on that X-ray.

In [ ]:
# ===== Step 1: split the data =====
# Put all the X-rays together, then make a random split: 80% for training and 20% for testing.
X = torch.cat([train_x, val_x, test_x])
Y = torch.cat([train_y, val_y, test_y])
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

# ===== Step 2: build the model =====
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 14),
    nn.Softmax(dim=1),  # turn the 14 outputs into probabilities
).to(device)
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ===== Step 3: decide how to measure quality =====
def accuracy(model, X, Y):
    predictions = predict(model, X) > 0.5  # a finding counts as present when its probability is above 0.5
    return (predictions == Y).float().mean().item()

# ===== Step 4: train for 5 epochs and keep the best model =====
best_accuracy = 0
for epoch in range(5):
    train_one_epoch(model, X_train, Y_train, loss_fn, optimizer)
    acc = accuracy(model, X_test, Y_test)
    print(f"Epoch {epoch + 1}: test accuracy {acc:.1%}")
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = copy.deepcopy(model)

print(f"Final result: {best_accuracy:.1%} accuracy on the test set. The model is ready for the hospital.")

---
## Phase 1: review without AI (12 minutes)

Double-click this cell to type in it. Write one row for each problem you find. The most important column is **what would go wrong in the hospital** if nobody fixed the problem.

| # | Where (step and line) | What is wrong | What would go wrong in the hospital | How you would fix it |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |

<details><summary><b>Stuck? Click here for four questions that every reviewer should ask.</b></summary>

1. **Where does the test data come from?** Could the model have seen something from the test set during training, even indirectly? Think about who the X-rays belong to.
2. **Does the last layer fit the task?** How many findings can one X-ray have at the same time? Can the last layer of this model express that?
3. **Would a lazy model also get a high score?** Imagine a model that ignores the image and always answers "no finding". What score would it get with this way of measuring?
4. **How often is the test set used?** The test set is meant to be a fair final exam. Is it used only once, at the end?
</details>

### Extra challenge: prove it with one line of code

Write one line of code that shows the 95% doesn't tell us much. The setup cell has already loaded `test_y`, the true labels of the official test X-rays, so you can run your line without training anything.

In [ ]:
# Your one line goes here.
# Hint: what accuracy would a model get if it said "no finding" for every finding on every X-ray?

---
## Phase 2: review with AI (8 minutes)

Now compare your review with an AI review. Copy the code cell above into Claude or ChatGPT and ask it to review the code. You decide how to ask, because the way you phrase the question is part of the exercise. Try at least two different questions, for example a short one ("Review this code.") and one that explains what the code is for and who will use the model.

Double-click to fill in this table.

| Question | Your answer |
|---|---|
| Which problems did the AI find that your team missed? | |
| Which problems did your team find that the AI missed, or explained wrongly? | |
| Did the AI say something with confidence that was wrong, or that didn't matter? | |
| Which way of asking gave the most useful review? | |

💬 **Discuss as a team:** suppose an AI had written this code for you, and you had asked another AI to check it. Would the problems have been caught? Which of the four problems could you only find because you know how hospital data is collected?

---
## Phase 3: your verdict (2 minutes)
Choose your answer in the form on the right and run the cell. Be ready to explain your choice to the room in one or two sentences.

In [ ]:
#@title 🗳️ Your team's verdict
verdict = "Fix it first, then test again"  #@param ["Use it as it is", "Fix it first, then test again", "Start over"]
the_worst_problem = ""  #@param {type:"string"}
one_sentence_for_the_hospital = ""  #@param {type:"string"}
print(f'Verdict: {verdict}\nThe worst problem: {the_worst_problem}\nWhat we tell the hospital: {one_sentence_for_the_hospital}')

---
## Solution
The instructor gives out the password at the end of this part. Try the review yourselves first: you learn most from the problems you find on your own.

In [ ]:
#@title 🔑 Show the solution (the instructor gives you the password at the end)
password = ""  #@param {type:"string"}
import base64, hashlib
from IPython.display import Markdown, display
BOX = {"salt": "ef7f5dda4bbb5f7b", "check": "4347f3ecad71a3849cc7f298b676b4e874da32fdc9563ab050d5c0dfa3fe47be", "data": "T2X3xrquRFhXjNphKg97NnmLsyW1itEX09y55NyP+kxQPu/WgHsINfrOS9thJke4YslRcckHWE5Drm4DgDMRA8DLef0jBNa6a1meUOaM6zcrIoygiwgymSO3L/pr7mUskJAOaApTK4G4fhmsEE9Iviwtf0kkdv18RETw/WAA048LOmqPikPJ51yhR84wmTWFr7Q0b8Pn8U9YUBr0p6FoZEb84r6ry/9LkMJ4hnDDl6KhJMTMfsymMAbDwHa5Afnl94NfQudYNNIvdofBv/wMQz5bT3hkPhWhnBH2fP6Z3V1wndoG7QzoQZfJa8NmvhbKyR8MbllckqS0IUgKSY6loyA13F2U5bDQJvGH0HArLhK+wmiwqYRNxBseGmlJVX/n/8O2WjNqO8/BMaAeU/sIdfbhNIvjUDpXhyrjqYBuvHTSJMUz3Yxx0pWw7rkQlK3d5eTJzsA9/c6jtmmzojnLQnIWTa9SEl1EfpvhWlE5s6r+9BfQ8GOzlZ7e64sUp2F0e48IRUGZemMtlOGItQwWJ1I3Pfcw8aVYl0SrlpRGo6ZAAR5WPS/n7vjcZKqoEdhTHwNcvuZi37rekRkXL/FhbDXCtnPQSR9oB2mAgkm2b7wUeL67Z2N4tRxI8kkY8uRjo3vIz0OD4LjQYWzpMEr9AOoZjAXwtrvdaTHBB2YSYOIwtfzamJD18mIe2mJaomRDCzrbCeU5BmihGSnJYJ1sOPZfZuox5W/DFPzqYxoBapQZizlVZUfcTmKU46mhJiSe52dXj88irIN5LkcF35tXxrk1C2KyS2hFGPU68hH8mTwX9Th+qdl3tUX5sdBPSqAP64Aj4L3tddv8RBjHFueLOJFO9GpN6Oj/PFMrcTS2UnpM/THEmS45pFUSZfclSbgPZbN1Vv9bcAP2a96qfL2Z9JcUYGsSJXF92+LEJRrKGbv7msyzm3VDu+SSS8CrP92pCtXRJoXBAdRoGJoEJw5ZB/1LXgIIDPmLeFXXsTdLj388c6VkymH41MWAq8/y/12xSt1caPiui0ECP6D/pDMRL2TmrsAbLsWDILL/dRBY2XkAtByUcWmUIvhWCmeP3bcV2XNG8FeQ/XrgpBcpnIvXuquV+pk75Td9dRzahzG9WGAPbQVFduzBoWmdpYQUPwHMbNkN/9r6XhoI10daR/adtTWrA/kIZku6G+UUJgdNjLDuuVly74pOUhh6p1uxRhScaUQDnV3KWyNIPJx/0waUh8MKyF6RjWwP0hdDj96LsJ69izdC1AE1mC+lHsq8PuJu6f2PbrrR9jflBTmh0EvmFFnurXUMKn4sMFD52bFxx95KINBjTCvi3M5k+D890erF1iPF0iHu5YMJFa+Q6r6+QSyxU4vaaLWdLE5tNL6REeOq5QihzLSHBg2mfMJgM9T2BM6SpZ0T00LgKcUxf5ZPbdY2JCD8I7nvUz0VjYQNW9XQtDhKEYEiaZcZtLigIP726DvJ7ftsuWyw/22XMhhUnixajPzBADQd0/2zyQZ5AlxVWKsDjFtlqjRy50lgkhuBvmQgjjCGQDpWJzIpoG9v0iK+hd8dX2RNYOvY6E3HchFp7BMweDzTiZ+3TM5c9jdwtP8akt+2dhqxkXy1L42HjUrdZPSDdc/KO68RRdGs+Gns1jwbi4kWkBeUedUcaQ9rdlOnHOiHNKOMO+wwjX/zuLxkEEp99sBXktibYJFW+iVbiQoE8e0HVpulhOnlh9dfgToUlXgb7qidqjbmqfJvEmX4z36gcbZfK/2zTxyy63TDp5AX/4vX3iYhEBgDOguEDUJPAE0TTaYMbAtaNOMs+og4XEbfrQWNRioufZi8qyrYr7vtKdBwkYF1Rh/PhVYZUY2019/A2tHKOhTxbKAIKmLknCIKJJ8jCiyFC02XofBOutoY0GGScPnbrx9rQOeeD0+idVJ+jPeWwmp3KRX/kqiDYDuBZC4nH4Fu4uki3284nnQef0rc3PelwLLGWPdLHwI9qrjX/kGZ1qRjlsFN9nCnCR1lrljvTisz946/dl5bC7SjMsbdjh4FNeZG21WcFTeIZiv2yziVUwXkyHuidb6+jluOTUFvcUh5FtYO6xDiebYj7Yl2xM3MGfzS/bwJZ9Vj4sLWvDu6qVmJFkECM/g2HGfkzrRw/d4qO0O6em2bM9b5Lt22nBh5lRAypoHH/Y6i0jyyp6WJHm6xJLAWGC6zlyjrxiblP6kJ7NNo0YFyNwFgKyW1wASeqy4H+CHb/LJwG/zqhvdye4suSqtfQzkdYltVbaAmPkW/1dRmW+qN6mNawYUEp7jvN19GwBDoVpSQuGVQas6hjutaWlkPxmmhi3padrsmvrK4TtcFakm2JS1TrKHK8zdiJF7V6gfvkn9QazJrZHLdO44kALR1aSgJtn3wIEyPQLwPhqiPQRz9pCoVZfQRWQDkja1nUPgq23TSCw4n+tSW7esvAT6ErANyyNDuA3gEgXcQEsCuTax4dVftwNOeVBnR3IAgJ/tJQUBbE1XRa7BTqv+YsYhmi7m0xffVDU3rFSueV5k4pV/RADS5SudJfoh8+ZVIISKD2NbrwQTUVwilZ812hZ02gStIX+1Czjm2/DDIg+gkLv98NaGhivgE2RVA1G9Zsb+K/U0nbbNd4qEsGfc5hTFSpL9GzeDRxaj03LWmHl7srJlqdNbehohmyBG9MVLcTx9Za9dVTvumOe7QkwLtwvux4lWkjj9yNVm5HFfE+wN2OIOypyRxMcWQ5QNyOt7T4rEhwTKv14wwHem83D3OhHmRUy8DgbzyKGps/Ux4zOBQIJaF868DN59ARt5h4gE0mNXTDZ+kAM6hjtu3PemgChqlc2LerzGzfg/3aOuwtMk8fOmzvPcYoSxoy2BBe+N/f953L4ExFgRnZztTwP3yJs+P1YVGRj6PgpJjP4lhRIfGjk3dgkUv0HI0Jn3eBQq+FPGPa7OWiFCXL6sALCJiB7aSHR3au6Xq+VsEDMkz/AWU8a5x7RxtnL7hzJvikrsXCl/fAzM1iWvUXlCWQBZS4GKYZC2EF5CMehsgpkEF6V4NZ9hKArpVbPphx5ThN3uLS7PtEIpsJs8EUbYY18nbpqTpq6EWcgyCEEToJlYIPR9Aqs1V7ZN/KBK5z7dMuiqrBoxMAm40f0gh+BUkm9/E2cft+HsNrOGlY2zSu/M+vKJVaI/tSU0iUqB+Z8v4x8H7esUQInq9F4a1I04TgRLRoRkCC6wYr5thet7lPmbGqVRhXOj+tnYd9hk0EQ+lRp6UhF3EAgzI/edQqruZjDfeEuDr6vSvEkE9/9bVAhqdSnEWY1NDtCar27n3ij/bj836p+2H1ET25YKMDYEe29lRPe3a5ZkCfNWc+SEbpcwcmuPh+6vtbMs457UOIzLtkgFvSA5wXJitOkWwQM9BfS9p8ajkmAMuT2W5uNXu3aVYy2GL71wGDzb4m7/5oVeDZ80OhyQIWOVAdWoWV3DVvt679pc3Lr710aVxlsshrgJ+VZ0e4h9j5/qneyc67QKkkGIYkViTTWiwlZ2rtdD6gaoADG/LaFWuYitC6jelozFLYrJgmxvOkbwt96BMxM687LZWV4JHm85QXHtHfwbjOsnp5jOkTrIGl66KJ00cKihKjDcNeU7OJs1PMMDNUJRq0etqRU3OV7PxOBzxM2xdXnFgvzYxfm+4i6FDmvMUOF2QYKmO9LkbqLp6ppleXIeE0keNFOVLoDwPiY7fgWxURvgRzkTTwW5HlYq355ODfK+duNPUuX9NUqiVnafyK6ysRlAw3z2MnG2tw5aMzEJXirUZsreSi3ioGFWcJQA8CiGdkzkPdphW8Li+YEIcX877d8UVTHxmv8WGHNjJaJfFjpe1AzEhlEQLm4RY7K6wSNdYBzQQav1R9K4S/i4v6NO9MKxaRr5L6Pvr2ga8l+kyO6gkLu9T5HWQ6nuhZJJ/ip80OItvOiGKdahVvkcq3ceu06QdxpcWheSS9g+keZpk8h4C+BnVOuvSZrwlhh5f0mvJkwLFKNQ97T61L0+HAnRqxNmKOpmknf8zRn+uv4YYzcTlPZfSuR6JD96ApaLJn5euPKrNAL33DP7PHHp6Y6Dmi6EbXmynASjQRD3vQvfiPKTSO8Vn3NCG/yzLXMDQSwVcG6Y7aNtDvdfE1FpRdoC3sdhP9mgKjem46Z8xt020qAMu9VR6oEx5eClwD7RrDP38SuyXpDkCedGEwmUMtfr4q4uDi8XK8MT8cBUoFpPVQl6VxUYt8tl6gEzKg8f4HYw03EmnPjPNShATTHh3L0/JrfWcCPOZL1wQsaT3kPu8zfz3uEgV8M8gK1bc/C1/y6suTYiK+1HYBbF1ls9W3QDh/SRtQEE3VDDXsgpmasbVRVpYmeBWOr4k//Juxi3Ek+rPn3KhSHtzjvJ9FiUii2YfP4MNjz2+47NrAq8EDdNq5W6hlQfk9RTfT32Q7Pa2zBMdRonqmtpvkR1/Xq+YPnd63MO1KI15Cf6+5qDkiXa37JRf9DCiE4ID288StaSPKxPHQ5o9kdilJKuqGE1FHIc0AgmkZPyL8bttrTb63pG0TjlbLRt+xDyCrlWfKqOVOL25NXI/S2PUfyARK2hEyEuTP5OoOs6ZJR0YF1UaJDssVZijGFt85FcJ0tniaJpSnKzC5BeuB7COzPfu6XlshUOIqIFnqu8D8rhRotQl5nmoft3v9qBHMeRTw9WM5fDHEIIOoBI8fdl2GpCoJEBAg7KI2YV86JLhHxxCeibvrI9nlRJg5Idve55twRNevQFmDQRQg1vIZDHSBav4UznE7Hi4tSg+l1r8HdglEeKXhUnfVAoS0Q+mEB/fVM0hlRk9u9EbSNwWb7H5sQzsQPzob1QIOxd6dEASnhJtOYvbQXUfGmNeVbr1wew2JDUI3pnX1KAN9YuASb960s6flpF6jMzRcLSmBR059CpKTrn1k1u/3ZCuBIzu8awSeja2fSAGpsBgSH7et5SXJEW8fR0t7H8NZ3on/jyNNtRGazSxXxIVWce+x4TFj5UtK4A6aD0a5uHYKPdWP4QZ/2erexin3mw8w67bKWsfRSxW4Ks6Z/1QWVGyVJGaW3IKltQmkq/lhoRkauMoUN4vKzQNi5B2/fJ4fGHzL+tmkOQBSBSZPQyfAZDmUASBhEdBMy6XWZlmhfCIGLqdVmlEgdQaeuuY1l9djXF1oo0EZvrI14pVmhFCUmQZixQRoOGv6eDGavTE3PLDvNADufD26wxpltZ8WhpluCrKYCkrWUea6/bztrs3x7XMeyR3JrtgM26gy8ZC2tNGulY4TLPyjbhQ0BbbAARrTfeWravetuoePAQ4SsmPmtkmkBD8l0nG58KXcX5GQ5Jk8DlmVWfHcZKflefWTaGxZgolGA5j+QWEUopSjFXc2EAUFHN0QCN1Z9Fa0Rk/gjyCO1O70JFHBCgQquRCTV/zn/61oXhELc9QDz30GZi1Yy2xQES5T22N1sBJWNNYmiFu8KILxNdA+k9M/bbSK8s45v6WKdhyKk+GwBlCbU6DZ2/8IdCgVzhcWpyXDzFWrFkMz7erHIu3X3U6Zsrg/DlZRvgrmtyehhBjlRJ4zgS0k53Pp8ZpPpb/oGfwruZGncP2Vs63G0VSwE5+r+IbS+WmEg2xChrWYBO5p2OC4r0fAoG2aG/LyhO9zyvhYTNrDMgRE20M+h7ya0JcaYm6av4QeX5BFbAb5MoZrP/0sCPRmZPIElS2eaMukucTDSj1Wzo8g7dM01LVKSpuV+0uil7eMVD5JIEr2lgCLyllU/DImK1Q3FkyOks/4KxqjT8SyFvFYHqaZom9XFBK5hoFRtDpgHn6Ix26vrgD/yNTwnv5dMY4TGwUM3Hn2mD0W2Zn16EHxLvYOFrszePL5h+xKoivSkM1nxU1KyRvzsygMvhChtrw0mT36kEfqOGbd2y8aT2Une6JUHBw6MElbEKUSHjlh7VU+KKDljpPP+L3rGFAyL0v9fvNbUVQh8M0MFRxs+PGaYeaag4oP0ByKhuNe+urueXnLZoaV8I4qejmeevmrO9PcyK1FZY/rxj3PM+VbUKoaHowa59KKifDV7zqgjFuz6/t2ijPKJiKuOmjQdhJHkMJZSEOrVm7MN0hkJFE7iicT6rNhgN7rW3HcCxDoFO0nx6aTM07RlTcNuzrTnZHQQ4cBm5zV1xO82Op2c1tzpXYptSWGY8NVWh8kALZ93w31aoodu4w1WXQ32tqsAikjCL80F+2HVVzNoSbGbZLnyRi8rXswToG9bHcHqb4NMjiG0kBAoleIcYjOtYiH6m40andSO4e3H1ZMpsSzOWbxaIR0wCzGNUkYN0Gtem7gt44QM3IkUcHOjfdYbwLikJWyG8cYVbaQKnMBWpfz4gW6DtyN56EZBT35zpEv4C/KU453LGBKuE+51CTWCKJkto5l9FTYQrpLQ9tRp0sxeieQ/Y11ZMFLEYp6YFxoyrXnWQ7Q7VKkmfr3eoqXH0kl//tz5w9F9QpkGTKuumyDuJhu9kYjvxAp3pxhRL6P7dvNqecLZfFjFLLYEfaZ9FJ6Q4zF9Fv2bvVjxRqvWR+f7vZo1oFVpLizKxLeelitcpFd6cf4HTXarhtLqYwFq8YsdR7UiqIljA4cVZnkTyNs+mEybOviRg3QrOlBcah5mUKh5wFsh6eeeioBAKXrg6hFXKNgZDixsPZs4hJ1JknXQWCONgesIJU8xyZeFR4uB35SJpdnPdk8RyPcswagWOVeW8Ca/2e6gT6aixjEAx5n+X0u3NebeSNxKwyhwXdR4HxEO0E/lLnBvDl6rXVGqBtAt+zdkMum2hgAfOB0l+YVSGHptsQQ360V/Mpg8x7cpSCvcAP6rwfbMFz83S6XqGizZlNmzNworlsS5KLb8guYNYyHYcksf58U7ZeS9a+xbywCYEA68XGmCEJXatC1mNW8q2Ap4xAlZEVizd2v0juhcOXt5xEg4JdU6Ph91dCGHSQCWM3gEY/jsl9/J4ytuFlSgKuegENoQrMSH33yYcN7box/xEIg/9LoWsnxPR2BwNurhKFo39tJth44E6TTGgGKrLZ9KachHDRWMkQcfSjqU1jNAtz8ywonq2f/peloP1gpIWD4BnsmHeK44OmMKiRMSxi13Cx+ln8HaL2qVPBc81ZfFSb4WhFfTauxmTLOqdj3xLpfuJDSwJfljCjbhGtIwnp1fdJ9QwZsea6g9Ycju6/J5wtXNckQKJetQBGYjQQ+KUbr/ikPSXU5Zv4aBnwAvD2buuvZ6hSh/169ppV0TFf+a2bEzSd1WGzVR/opJcb0j7pXAj46qkmTDmKZeF+XJnzqtVfg3WqixYANPDrf+38lnjcwzkvuHVyqnH7CYbjaKHc0iaMw3o0yTzaiJQmUjDS/rMRfBgTAgNArJjzOScXfkfqWzxy2C9o8yyUWr4ddAFPZny1dkHRnFZC1n5aTyZd//82GLNO7B7pH+14QSSjNYNfPlk9o+cE7ki3WeE9gBgUt4tkajFuP4SDDezw5b7IeclaqoZg+8V8l3DrJ1RYbc0Mir2WwsBHGv/OFemKKGvJjv9y4XSArChI9pMRGIKxxu8uoXeJXHU58/woPkThuk7sGDH7Yn47XQzgoqFN2ReOwr0dBVOn2Iy842VE2QtV4v6s3zeM8mgqBBItMU3GHa3tUQyOTVU+VFkWsPhMp3vVjyzydTnI2X7mvUo+s965YmSktj0IRODpHAFXGI6BSE6S5Uz/zaksjtyMnGyIRjgQwhFBcA=="}

def _keystream(salt, pw, n):
    out, i = b"", 0
    while len(out) < n:
        out += hashlib.sha256(f"{salt}:{pw}:{i}".encode()).digest()
        i += 1
    return out[:n]

pw = password.strip().lower()
if not pw:
    print("Type the password in the field on the right and run this cell again.")
elif hashlib.sha256(f"{BOX['salt']}:check:{pw}".encode()).hexdigest() != BOX["check"]:
    print("That is not the right password. Check the spelling: it is all lowercase, with dashes.")
else:
    cipher = base64.b64decode(BOX["data"])
    text = bytes(a ^ b for a, b in zip(cipher, _keystream(BOX["salt"], pw, len(cipher)))).decode("utf-8")
    display(Markdown(text.replace("__CONFIG__", globals().get("CONFIG_ID", "?"))))